# Anticipación en semanas 2, 4 y 6: utilidad operativa potencial

Preparado hasta que finalice la etapa `anticipacion`. Cada corte usa su propia cohorte elegible y conserva los folds por estudiante del día 14, sin condicionar pertenencia a estar activo al día 56. Se purgan estudiantes presentes en cualquier corte de 2014J sin usar sus outcomes.

Se comparan ambas familias finalistas con los mismos parámetros longitudinales fijados: solo estáticas, estáticas más indicador de registros y longitudinal completo. Es ablación de información en desarrollo, no tuning independiente de cada semana ni evidencia causal.


In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset_3/config/protocolo-v2.json').is_file(): ROOT=here/'pset_3'
assert ROOT is not None, 'Ejecutar desde pset_3 o notebooks; revisar README'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


Protocolo verificado: 07111fdf724a0cd8936ea5abc0ff983769f15d0f6216b24172c56aaf18e8ab0d
Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.


In [2]:
early=report('docs/anticipacion/resultados.json')
assert early['status']=='complete' and early['protocol_id']==plan['id']
assert early['holdout_used'] is False and early['winner_changed'] is False
display(pd.DataFrame(early['cohorts']))
display(pd.DataFrame([{k:v for k,v in r.items() if k!='folds'} for r in early['models']]))


,cutoff_day,n,unique_students,fail,no_activity
0,14,17246,16202,4914,1141
1,28,16838,15866,4914,654
2,42,16529,15626,4914,510
3,56,16216,15371,4914,432


,family,cutoff_day,mode,run_id,plan_id,ap_fail_mean,ap_fail_std,coverage_mean,coverage_std,train_ap_fail_mean,train_ap_fail_std,fit_seconds_mean,fit_seconds_std,inference_seconds_mean,inference_seconds_std,api_cost_accounted_usd
0,boosting,14,estatico,096e4f590ff04551b5d8fa3d196713a7,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.396542,0.013512,0.153581,0.006849,0.449589,0.032034,1.233971,0.282594,0.094976,0.029511,0.0
1,boosting,14,indicador,a943fcd65f104fea8f38ac2ba7361bfa,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.424690,0.015509,0.174347,0.006731,0.462175,0.030680,1.404160,0.553313,0.128090,0.079067,0.0
2,boosting,14,longitudinal,a6c0b65d4e354ceaabfc565c88bfa0c2,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.456963,0.010680,0.186278,0.006426,0.495503,0.019126,1.592283,0.293712,0.098732,0.005250,0.0
3,lineal,14,estatico,73257c1514b24567876e1b3237c405a8,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.391673,0.014024,0.159718,0.009581,0.409421,0.029928,0.294673,0.054442,0.052344,0.030281,0.0
4,lineal,14,indicador,4ff4c9cc6ca34be98283f3eec5b1f211,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.417165,0.014231,0.175008,0.008624,0.429039,0.030606,0.313445,0.138083,0.046403,0.007361,0.0
5,lineal,14,longitudinal,bbeb3ad044f748e085517f0bfcb97f72,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.451363,0.010121,0.186841,0.008984,0.459079,0.019491,0.346864,0.018360,0.057537,0.010412,0.0
6,boosting,28,estatico,6e0b3d2b12cf481eb45221c7349b3dcb,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.408455,0.013616,0.154396,0.007083,0.457093,0.024183,1.421490,0.322175,0.093056,0.012893,0.0
7,boosting,28,indicador,08e32f6352f04edf9d031f7af36a49d1,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.443550,0.016472,0.178208,0.009008,0.492339,0.027625,1.809783,0.361820,0.119610,0.035421,0.0
8,boosting,28,longitudinal,048df6920ba747d28d957b052e685666,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.492690,0.010697,0.201457,0.007693,0.548055,0.017833,2.151061,0.955497,0.144039,0.034891,0.0
9,lineal,28,estatico,249abdfbd6f3402b8613925073351955,7deb66f98a2443a6406d48d4fe62ed8d7e219ec82a63ec...,0.402216,0.014082,0.158509,0.009375,0.422655,0.027269,0.265104,0.055367,0.052377,0.026037,0.0


## Un presupuesto compartido entre rondas

K nace de la cohorte del día 14, por edición y fold. Se contrastan una sola ronda en semana 2, 4 o 6 y una política escalonada con 40/30/30 % del mismo presupuesto. No se repiten matrículas; también se informan personas únicas. Se muestran escenarios de 7 días de coordinación y 14 de apoyo, con plazos ilustrativos en los días 56 y 70. No son políticas institucionales validadas.


In [3]:
policies=pd.DataFrame([{k:v for k,v in p.items() if k!='rounds'} for p in early['policies']])
assert (policies['used']<=policies['budget']).all()
display(policies)
print('Los TP con margen son candidatos potencialmente atendibles bajo los supuestos, no fracasos evitados por una tutoría.')


,n_base_14,fail_base_14,budget,used,tp,fp,coverage,precision,unique_students,mean_lead_weeks,tp_with_margin_day56,tp_with_margin_day70,family,policy
0,17246,4914,1691,1691,755,936,0.153643,0.446481,1626,33.760328,755,755,boosting,estatico_semana2
1,17246,4914,1691,1691,915,776,0.186203,0.541100,1652,33.760328,915,915,boosting,longitudinal_semana2
2,17246,4914,1691,1691,1011,680,0.205739,0.597871,1633,31.760328,1011,1011,boosting,longitudinal_semana4
3,17246,4914,1691,1691,1088,603,0.221408,0.643406,1631,29.760328,0,1088,boosting,longitudinal_semana6
4,17246,4914,1691,1691,1017,674,0.206960,0.601419,1633,31.878601,677,1017,boosting,longitudinal_escalonada
5,17246,4914,1691,1691,785,906,0.159748,0.464222,1641,33.760328,785,785,lineal,estatico_semana2
6,17246,4914,1691,1691,918,773,0.186813,0.542874,1647,33.760328,918,918,lineal,longitudinal_semana2
7,17246,4914,1691,1691,1015,676,0.206553,0.600237,1641,31.760328,1015,1015,lineal,longitudinal_semana4
8,17246,4914,1691,1691,1101,590,0.224054,0.651094,1639,29.760328,0,1101,lineal,longitudinal_semana6
9,17246,4914,1691,1691,1028,663,0.209198,0.607924,1647,31.878601,694,1028,lineal,longitudinal_escalonada


Los TP con margen son candidatos potencialmente atendibles bajo los supuestos, no fracasos evitados por una tutoría.


## Ausencia de actividad: asociación e incremento predictivo

Una tasa alta de Fail en el grupo sin registros no basta para demostrar utilidad incremental. Se muestran diferencia de riesgo, razón de riesgos, correlación binaria phi, intervalos por remuestreo de estudiante y estandarización por edición. El incremento predictivo se evalúa separadamente en la comparación estáticas / estáticas+indicador / longitudinal. Estandarizar no demuestra ausencia de confusores ni causalidad.


In [4]:
display(pd.DataFrame(early['associations']))
from IPython.display import display, Markdown
for limitation in early['limits']:
    display(Markdown('- '+limitation))


,cutoff_day,no_record_n,no_record_fail,record_n,record_fail,risk_no_record,risk_record,risk_difference,risk_ratio,phi_binary,risk_difference_ci95,risk_ratio_ci95,standardized_risk_difference,strata_included,standardization_population_fraction,scope
0,14,1141,601,16105,4313,0.526731,0.267805,0.258926,1.966845,0.142582,"[0.22830911075093263, 0.28842679730118675]","[1.8472565689098872, 2.0801505401167497]",0.262743,14,0.980517,Observed development association; standardizat...
1,28,654,420,16184,4494,0.642202,0.277682,0.364520,2.312727,0.154926,"[0.3301159553332403, 0.4028214792604457]","[2.1755961668119017, 2.4632000734034603]",0.375365,14,0.980045,Observed development association; standardizat...
2,42,510,356,16019,4558,0.698039,0.284537,0.413502,2.453245,0.156442,"[0.3718344420962701, 0.4494650720563911]","[2.296054695196426, 2.5948937837924246]",0.444123,14,0.979733,Observed development association; standardizat...
3,56,432,327,15784,4587,0.756944,0.290611,0.466334,2.604668,0.163400,"[0.4256290629420261, 0.5080426585273845]","[2.4546360712180726, 2.753730034742874]",0.489063,14,0.979341,Observed development association; standardizat...


- exploratory reused development CV; no causal intervention estimate

- same hyperparameters across early variants, not independent early tuning

- policy ranks within edition/fold, one shared K based on day14; enrollment and unique-student units differ

- 21-day margin is an illustrative operational scenario, not an institutional deadline

- early label-maturity and publication lag prevent prospective certification